# Fine-tune the LLM Judge (LoRA/DoRA, LambdaRank objective)

The `llm_yesno` Qwen judge is the ensemble's #1 signal *and* its main failure (false negatives, §8a),
and it is the **only unsupervised component** in an otherwise fine-tuned pipeline (§9e idea #1). CoT
prompting was dev-null (§11a) → the lever is **supervision**, not more zero-shot cleverness.

## Design (deliberate, 80GB A100)
- **bf16 LoRA (DoRA-enabled), not QLoRA.** 80GB fits the bf16 base (~15GB) + adapters with headroom;
  no 4-bit quantization loss. `USE_DORA=True` = weight-decomposed LoRA (stronger variant).
- **LambdaRank-weighted pairwise loss on the yes/no margin** (not naive pointwise). We consume the
  judge as a scalar feature whose *within-topic ordering* is what the ensemble uses; a |ΔNDCG|-weighted
  pairwise loss trains exactly that and ties the judge to the metric. `LOSS='pointwise'` = CE fallback.
- **Graded labels (eligible=2 / excluded=1 / not=0).** Training the margin to rank eligible>excluded>not
  directly attacks the rel=1 eligibility-vs-topicality mismatch (§9e idea #2) inside the judge.
- **Hard negatives** = dense-retriever-top rel=0 judged docs per topic (as in reranker-v2).
- **Interface unchanged:** score = logsumexp(yes) − logsumexp(no) at the gen position, same prompt as
  the zero-shot judge → drop-in replacement for the `llm_yesno` ensemble feature.
- **Anti-gaming:** train on TREC21+KZ; dev = held-out TREC21 topics for checkpoint selection;
  TREC22/23 never touched here.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers peft accelerate bitsandbytes datasets sentence-transformers pytrec_eval tqdm
# peft hard-requires torchao>=0.16.0 inside PeftModel/get_peft_model; Colab often ships 0.10.0,
# which raises ImportError. Upgrade LAST so nothing re-resolves it down, then RESTART the runtime.
!pip install -q -U "torchao>=0.16.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT       = '/content/drive/MyDrive/ct_data23'
EVAL_ROOT       = f'{DATA_ROOT}/evaluation'
TREC_ROOT       = f'{EVAL_ROOT}/trec_data'
KZ_ROOT         = f'{EVAL_ROOT}/kz_data'
FULLTEXT_CORPUS = f'{DATA_ROOT}/doc_texts_fulltext.txt'
DENSE_EMB       = f'{DATA_ROOT}/doc_embeddings_retriever-v2_fulltext.npy'
RETRIEVER       = 'semaj83/ctmatch-retriever-v2'
BASE_MODEL      = 'Qwen/Qwen2.5-7B-Instruct'
OUT_DIR         = f'{DATA_ROOT}/judge_lora_v1'

TRAIN_SOURCES   = ['trec21', 'kz']     # KZ pairs are valid supervision (its degeneracy was an EVAL artifact)
DEV_FRAC        = 0.15                 # held-out TREC21 topics for checkpoint selection
LOSS            = 'lambdarank'         # 'lambdarank' (primary) | 'pointwise' (CE fallback ablation)
USE_DORA        = True
GROUP_POS_CAP   = 8                    # max positives per topic group
GROUP_SIZE      = 16                   # pos + hard-neg per topic group
MAX_LEN         = 1024                 # patient + truncated full-text trial (raise to 1536 if memory allows)
EPOCHS          = 2                    # v1 run: epoch 2 best (dev 0.8713); epoch 3 overfit (0.8602). ~57 min/epoch.
LR              = 1e-4
ACCUM_TOPICS    = 4                    # gradient accumulation over topic-groups
SIGMA           = 1.0
os.environ['CTMATCH_DATA_ROOT'] = DATA_ROOT
os.makedirs(OUT_DIR, exist_ok=True)
print('config set | loss =', LOSS, '| DoRA =', USE_DORA)

In [ ]:
import numpy as np, random
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from ctmatch.evaluation.eval_utils import load_eval_datasets

random.seed(42)
idx = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
corpus_ids = [r['text'].strip() for r in idx]
corpus_txt = [l.rstrip('\n') for l in open(FULLTEXT_CORPUS)]
id2txt = dict(zip(corpus_ids, corpus_txt)); id2i = {d: i for i, d in enumerate(corpus_ids)}
doc_emb = np.load(DENSE_EMB).astype(np.float32); doc_emb /= (np.linalg.norm(doc_emb, axis=1, keepdims=True) + 1e-9)
enc = SentenceTransformer(RETRIEVER)

sets = load_eval_datasets(TREC_ROOT, KZ_ROOT)
sets = {k: v for k, v in sets.items() if k in TRAIN_SOURCES}

# build topic groups: positives (graded) + dense-mined hard negatives
def build_groups(names):
    groups = []
    for nm in names:
        ds = sets[nm]
        for tid, text in ds['topic2text'].items():
            rel = ds['rel_dict'].get(tid, {})
            judged = [(d, r) for d, r in rel.items() if d in id2i]
            pos = sorted([(d, r) for d, r in judged if r >= 1], key=lambda x: -x[1])[:GROUP_POS_CAP]
            negs = [d for d, r in judged if r == 0]
            if not pos or not negs:
                continue
            qv = enc.encode(text, normalize_embeddings=True).astype(np.float32)
            negs = sorted(negs, key=lambda d: -float(doc_emb[id2i[d]] @ qv))   # hardest (dense-top) first
            hard = negs[:max(1, GROUP_SIZE - len(pos))]
            docs = [(d, float(r)) for d, r in pos] + [(d, 0.0) for d in hard]
            groups.append({'topic_id': f'{nm}:{tid}', 'source': nm, 'text': text, 'docs': docs})
    return groups

all_groups = build_groups(TRAIN_SOURCES)
# dev split on TREC21 topics only
t21 = [g for g in all_groups if g['source'] == 'trec21']
random.shuffle(t21)
n_dev = max(1, int(len(t21) * DEV_FRAC))
dev_ids = {g['topic_id'] for g in t21[:n_dev]}
train_groups = [g for g in all_groups if g['topic_id'] not in dev_ids]
dev_groups   = [g for g in all_groups if g['topic_id'] in dev_ids]
print(f'groups: {len(all_groups)} total | train {len(train_groups)} | dev {len(dev_groups)} (held-out TREC21)')

In [ ]:
# ── OPTIONAL: synthetic narrative groups (gen_synthetic_pairs.ipynb) ─────────────────────────
# Secondary consumer (§11c: the pointwise-eligibility axis the judge feeds is near-saturated; the
# live SOTA lever is monoT5 v7). Each verified synthetic patient becomes ONE listwise group:
#   positive = its home trial (graded rel 2 or 1; text injected into id2txt under a '__SYN__' key)
#   negatives = dense-mined real-corpus hard negatives for that patient (reuses the exact machinery
#               build_groups uses for real topics) — so we do NOT need to generate rel=0.
# Appended to TRAIN only; dev stays real held-out TREC21 (never read success off synthetic data).
# No-op if the file is absent.
import json
SYNTH_PAIRS_PATH = f'{DATA_ROOT}/synthetic_pairs.jsonl'   # set to '' to skip
# Real train is only ~100-150 groups (TREC21-minus-dev + KZ). Keep synthetic a MINORITY-to-parity
# augmentation, not a takeover, on this near-saturated axis; raise deliberately if an A/B says to.
SYNTH_MAX        = 400      # cap synthetic groups; None = all

def _synth_trial_text(f):
    """Full-text-ish trial string from a corpus-schema field dict, comparable to id2txt corpus lines."""
    t = f.get('brief_title') or f.get('official_title') or ''
    c = f.get('conditions', '') or ''
    if isinstance(c, list): c = ', '.join(str(x) for x in c if x)
    s = f.get('brief_summary', '') or ''
    e = f.get('eligibility', '') or ''
    return f"{t}. conditions: {c}. {s} eligibility: {e}".strip()

if SYNTH_PAIRS_PATH and os.path.exists(SYNTH_PAIRS_PATH):
    syn = [json.loads(l) for l in open(SYNTH_PAIRS_PATH)]
    syn = [r for r in syn if r.get('verified') and r.get('rel', 0) >= 1]
    random.shuffle(syn)
    if SYNTH_MAX:
        syn = syn[:SYNTH_MAX]
    # batch-encode patient notes once, then dense-mine negatives per patient
    qv = enc.encode([r['patient_note'] for r in syn], normalize_embeddings=True,
                    batch_size=64, show_progress_bar=True).astype(np.float32)
    syn_groups = []
    for k, r in enumerate(syn):
        home_key = f"__SYN__{r['syn_id']}"
        id2txt[home_key] = _synth_trial_text(r['fields'])         # inject home-trial text
        sims = doc_emb @ qv[k]
        cand = np.argpartition(-sims, GROUP_SIZE + 5)[:GROUP_SIZE + 5]
        cand = cand[np.argsort(-sims[cand])]
        negs = [corpus_ids[i] for i in cand if corpus_ids[i] != r['nct_id']][:GROUP_SIZE - 1]
        docs = [(home_key, float(r['rel']))] + [(d, 0.0) for d in negs]
        syn_groups.append({'topic_id': f"syn:{r['syn_id']}", 'source': 'synthetic',
                           'text': r['patient_note'], 'docs': docs})
    train_groups += syn_groups
    print(f'+synthetic: {len(syn_groups)} groups appended to train '
          f'(train {len(train_groups)} | dev {len(dev_groups)} unchanged, real TREC21)')
else:
    print(f'no synthetic pairs at {SYNTH_PAIRS_PATH} — real topics only')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

dev = 'cuda'
tok = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16,
                                             attn_implementation='sdpa').to(dev)
model.config.use_cache = False
model.gradient_checkpointing_enable(); model.enable_input_require_grads()
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, use_dora=USE_DORA, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora); model.print_trainable_parameters()

YES = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ['yes','Yes',' yes',' Yes']})
NO  = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ['no','No',' no',' No']})
SYS = 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'
def prompt(patient, trial):
    u = (f'You are a clinical trial matching expert.\n\nPatient:\n{patient}\n\n'
         f'Trial:\n{trial[:1800]}\n\nIs this patient likely eligible for this trial? '
         'Answer with a single word: yes or no.')
    return tok.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':u}],
                                   tokenize=False, add_generation_prompt=True)

def score_group(patient, trials):
    prompts = [prompt(patient, t) for t in trials]
    e = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(dev)
    last = model(**e).logits[:, -1, :]                       # left-padded → gen position
    return torch.logsumexp(last[:, YES], -1) - torch.logsumexp(last[:, NO], -1)

In [ ]:
import torch.nn.functional as F

def lambdarank_loss(scores, labels, sigma=SIGMA, k=10):
    # scores (G,) grad; labels (G,) graded. |dNDCG|-weighted RankNet over pairs where y_i > y_j.
    ranks = torch.empty_like(scores); ranks[torch.argsort(scores, descending=True)] = torch.arange(len(scores), device=scores.device).float()
    gains = (2.0 ** labels - 1)
    disc = 1.0 / torch.log2(ranks + 2)
    ideal = torch.sort(labels, descending=True).values
    kk = min(k, len(ideal))
    idcg = ((2.0**ideal[:kk] - 1) / torch.log2(torch.arange(2, 2+kk, device=scores.device).float())).sum().clamp(min=1e-6)
    S = scores.unsqueeze(1) - scores.unsqueeze(0)
    pos = (labels.unsqueeze(1) > labels.unsqueeze(0)).float()
    dgain = (gains.unsqueeze(1) - gains.unsqueeze(0)).abs()
    ddisc = (disc.unsqueeze(1) - disc.unsqueeze(0)).abs()
    delta = (dgain * ddisc / idcg).detach()
    loss = (pos * delta * F.softplus(-sigma * S)).sum() / pos.sum().clamp(min=1)
    return loss

def pointwise_loss(scores, labels):
    # CE fallback: margin -> P(yes); target 1 if rel>=1 else 0
    return F.binary_cross_entropy_with_logits(scores, (labels >= 1).float())

def ndcg10(scores, labels):
    order = torch.argsort(torch.tensor(scores), descending=True)
    lab = torch.tensor(labels)[order][:10].float()
    dcg = ((2**lab - 1) / torch.log2(torch.arange(2, 2+len(lab)).float())).sum()
    ideal = torch.sort(torch.tensor(labels).float(), descending=True).values[:10]
    idcg = ((2**ideal - 1) / torch.log2(torch.arange(2, 2+len(ideal)).float())).sum().clamp(min=1e-6)
    return float(dcg / idcg)

In [ ]:
import bitsandbytes as bnb
from tqdm.auto import tqdm

opt = bnb.optim.PagedAdamW8bit([p for p in model.parameters() if p.requires_grad], lr=LR)
loss_fn = lambdarank_loss if LOSS == 'lambdarank' else pointwise_loss

@torch.no_grad()
def dev_ndcg():
    model.eval(); vals = []
    for g in dev_groups:
        s = score_group(g['text'], [id2txt[d] for d, _ in g['docs']]).float().cpu().tolist()
        vals.append(ndcg10(s, [y for _, y in g['docs']]))
    model.train(); return float(np.mean(vals))

print(f'dev NDCG@10 (zero-shot, pre-FT): {dev_ndcg():.4f}')
best = -1
for ep in range(EPOCHS):
    random.shuffle(train_groups); opt.zero_grad(); running = 0.0
    for i, g in enumerate(tqdm(train_groups, desc=f'epoch {ep+1}')):
        s = score_group(g['text'], [id2txt[d] for d, _ in g['docs']])
        y = torch.tensor([y for _, y in g['docs']], device=dev, dtype=torch.float)
        loss = loss_fn(s, y) / ACCUM_TOPICS
        loss.backward(); running += loss.item()
        if (i + 1) % ACCUM_TOPICS == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); opt.zero_grad()
    d = dev_ndcg()
    print(f'epoch {ep+1}: train_loss~{running/len(train_groups)*ACCUM_TOPICS:.4f}  dev NDCG@10 {d:.4f}')
    if d > best:
        best = d; model.save_pretrained(OUT_DIR); tok.save_pretrained(OUT_DIR)
        print(f'  ✓ saved best (dev {d:.4f}) -> {OUT_DIR}')
print(f'\nbest dev NDCG@10 {best:.4f} (vs zero-shot baseline printed above)')

## Use it: regenerate the `llm_yesno` feature, then re-run the ensemble

1. Load `BASE_MODEL` + this LoRA adapter (`OUT_DIR`); score with the **same** margin/prompt.
2. Write to a **new** file `llm_scores_ft.jsonl` (keep the zero-shot `llm_reranker_scores.jsonl` for A/B).
3. Point `train_ensemble_full.ipynb` `LLM_SCORES` at the fine-tuned file, retrain the LambdaMART on
   TREC21 CV, and compare. **Only after it wins on TREC21 CV**, run TREC22 once (anti-gaming).

Optional (graded readout): also emit `2*P(eligible)+1*P(excluded)` if a 3-way variant is trained;
the current binary-margin readout keeps the ensemble interface identical.

In [ ]:
# ── RELOAD the fine-tuned judge in a fresh runtime (notebook disconnected) ──────────────
# Run order for a reload session: cell-install, cell-drive, cell-config, cell-data, THIS cell.
# Do NOT run cell-model (it builds a fresh *untrained* LoRA and would overwrite the adapter).
# The trained epoch-2 adapter lives on Drive at OUT_DIR; nothing was lost with the disconnect.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

dev = 'cuda'
tok = AutoTokenizer.from_pretrained(OUT_DIR, padding_side='left')     # tokenizer was saved beside the adapter
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16,
                                            attn_implementation='sdpa').to(dev)
model = PeftModel.from_pretrained(base, OUT_DIR).to(dev)              # loads the epoch-2 best adapter (DoRA-aware)
model.config.use_cache = True                                        # inference config: cache on, no grad-checkpoint
model.eval()

YES = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ['yes','Yes',' yes',' Yes']})
NO  = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ['no','No',' no',' No']})
SYS = 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'
def prompt(patient, trial):
    u = (f'You are a clinical trial matching expert.\n\nPatient:\n{patient}\n\n'
         f'Trial:\n{trial[:1800]}\n\nIs this patient likely eligible for this trial? '
         'Answer with a single word: yes or no.')
    return tok.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':u}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def score_group(patient, trials):
    prompts = [prompt(patient, t) for t in trials]
    e = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(dev)
    last = model(**e).logits[:, -1, :]                               # left-padded → gen position
    return torch.logsumexp(last[:, YES], -1) - torch.logsumexp(last[:, NO], -1)
print('fine-tuned judge reloaded from', OUT_DIR)

In [ ]:
# Inference helper (run in a fresh session or reuse `model` above, now LoRA-adapted).
@torch.no_grad()
def judge_score(patient, trial):
    model.eval()
    return float(score_group(patient, [trial])[0])

# sanity: a dev topic's judged docs should rank pos above neg
g = dev_groups[0]
for d, y in g['docs'][:6]:
    print(f"rel={y:.0f}  score={judge_score(g['text'], id2txt[d]):+.2f}  {d}")